# 01 — Data Exploration

**Week:** 3

Goal: Load raw files in Databricks, inspect schemas, count rows, identify nulls and profile early issues.


In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/workspace/default/my_files/staymetrics/working_data"

bookings = spark.read.parquet(f"{VOLUME}/bookings.parquet")
guests = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOLUME}/guests.csv")
rooms = spark.read.option("multiLine", True).json(f"{VOLUME}/rooms.json")
rate_plans = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOLUME}/rate_plans.csv")
room_nights = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOLUME}/room_nights.csv")

bookings.createOrReplaceTempView("staymetrics_bookings")
guests.createOrReplaceTempView("staymetrics_guests")
rooms.createOrReplaceTempView("staymetrics_rooms")
rate_plans.createOrReplaceTempView("staymetrics_rate_plans")
room_nights.createOrReplaceTempView("staymetrics_room_nights")

print("StayMetrics: all 5 datasets loaded successfully.")

In [0]:
print("Bookings:", bookings.count())
print("Guests:", guests.count())
print("Rooms:", rooms.count())
print("Rate Plans:", rate_plans.count())
print("Room Nights:", room_nights.count())

display(bookings.limit(5))

In [0]:
print("===== BOOKINGS =====")
bookings.printSchema()

print("===== GUESTS =====")
guests.printSchema()
5 rows
4.101s runtime
$0

print("===== ROOMS =====")
rooms.printSchema()

print("===== RATE PLANS =====")
rate_plans.printSchema()

print("===== ROOM NIGHTS =====")
room_nights.printSchema()

In [0]:
print("===== BOOKING STATUS =====")
display(
    bookings.groupBy("booking_status")
    .count()
    .orderBy(F.desc("count"))
)

print("===== CHANNEL =====")
display(
    bookings.groupBy("channel")
    .count()
    .orderBy(F.desc("count"))
)

print("===== ROOM TYPE =====")
display(
    rooms.groupBy("room_type")
    .count()
    .orderBy(F.desc("count"))
)

print("===== MISSING VALUES =====")

checks = [
    ("booking_id", bookings.filter(F.col("booking_id").isNull()).count()),
    ("guest_id in bookings", bookings.filter(F.col("guest_id").isNull()).count()),
    ("rate_plan_id", bookings.filter(F.col("rate_plan_id").isNull()).count()),
    ("guest_id in guests", guests.filter(F.col("guest_id").isNull()).count()),
    ("room_id", rooms.filter(F.col("room_id").isNull()).count()),
    ("room_night_id", room_nights.filter(F.col("room_night_id").isNull()).count())
]

display(spark.createDataFrame(checks, ["check", "null_count"]))

In [0]:
bronze_table = "workspace.default.staymetrics_week03_bronze_demo_bookings"

(
    bookings
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit("bookings.parquet"))
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

bronze = spark.table(bronze_table)

print("Source rows :", bookings.count())
print("Bronze rows :", bronze.count())

display(bronze.limit(10))

In [0]:
%fs
ls /Volumes/workspace/default/my_files/staymetrics/

## Evidence to Capture

- Screenshot of data loaded in Databricks
- Screenshot of schema
- Screenshot of row count / basic profile
- Update `weekly_logs/week03_log.md`
